# What is PyTorch — Tensors + Iris Classifier – Solution

**Short name (GitHub):** `PTIntro`

**Lab source:** Codefinity / course notes *What is PyTorch* (tensors, creation helpers, `nn.Module`, training loop, Iris challenge).

Worked answers for `PTIntro_Practice_Skeleton.ipynb`. Helper functions live in `PTIntro.py`.

**Files you will use**
- `data/iris.csv` — 150 flowers × 4 measurements + `species`
- `ptintro_flowchart.png` — desired outcome
- `PTIntro_Cheatsheet.docx` — one-page lookup while you code
- `PTIntro.py` — split / model / train helpers for later cells


## Inline cheat-sheet (keep this cell visible)

See also **`PTIntro_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Tensor ranks | scalar 0-D, vector 1-D, matrix 2-D, batch of images 4-D `(N,C,H,W)` |
| From a list | `torch.tensor([[1,2],[3,4]])` — always copies |
| From NumPy | `torch.from_numpy(a)` shares memory; `torch.as_tensor(a)` may share |
| Zeros / ones | `torch.zeros(2,3)`, `torch.ones_like(x)` |
| Sequences | `torch.arange(start, end, step)` end **exclusive**; `torch.linspace(a,b,steps)` both ends **inclusive** |
| dtypes | features `float32`; class indices `long` / `torch.int64` |
| Device | `x.to("cuda")` if a GPU is present; this lab stays on CPU |
| Module | subclass `nn.Module`, call `super().__init__()`, store layers as `self.*` |
| Forward | `model(x)` — **not** `model.forward(x)` — so hooks run |
| ReLU hidden | `F.relu(self.fc1(x))` then linear logits |
| Loss | `nn.CrossEntropyLoss()` = log-softmax + NLL. Do **not** softmax first |
| Train step | `zero_grad()` → forward → loss → `backward()` → `step()` |
| Eval | `model.eval()` **and** `with torch.no_grad():` |
| Labels | `torch.argmax(logits, dim=1)` |

**Order that matters:** `zero_grad` before `backward`. `eval()` does not turn off autograd by itself.


## Desired outcome

![flowchart](ptintro_flowchart.png)

1. Create tensors from lists and from factory helpers (`zeros`, `ones`, `arange`, `linspace`, `*_like`).
2. Load Iris, encode species as `0/1/2`, split 80/20.
3. Wrap features as `float32` tensors and labels as `long`.
4. Define `IrisModel`: `4 → hidden ReLU → 3` logits.
5. Train with Adam + cross-entropy (full batch is fine — 120 rows).
6. `model.eval()` + `no_grad` + `argmax` → test accuracy.
7. Replay with Sequential / SGD, then turn the knobs in the simulation cell.


## Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# reproducibility for the whole notebook
torch.manual_seed(42)
np.random.seed(42)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("device will stay on CPU for this lab")


## 0. What is PyTorch (and why not just NumPy)?

PyTorch is a Python library for **tensors + autograd + neural-net modules**.

- **Dynamic graph ("define-by-run")** — the computation graph is built as Python runs. `if`/`for` in `forward` are normal Python.
- **GPU path** — the same tensor API runs on CUDA with `.to("cuda")`.
- **`torch.nn`** — layers, losses, `Module` containers.
- **`autograd`** — `loss.backward()` fills `.grad` on every parameter that `requires_grad=True`.

TensorFlow is the other major framework. Rough split that still holds in 2026: PyTorch dominates research and most new teaching material; TensorFlow remains strong in some production / TFX / Lite stacks. You do not need both for this lab.

Tensors are just n-dimensional arrays:

- 0-D scalar `5.0`
- 1-D vector `[1, 2, 3]`
- 2-D matrix `[[1, 2], [3, 4]]`
- 3-D stack of matrices (a tiny "batch of 2-channel images", a video slice, …)

Under the hood they are a contiguous buffer + shape + dtype + device. Indexing and slicing match NumPy. The extras that NumPy does not have are GPU placement and the tape that records ops for backprop.


## 1. Create tensors from Python lists

**Task.** Import is already done. Create:

1. A 2-D tensor from `[[1, 2], [3, 4]]`.
2. A 3-D tensor **directly from a nested list literal** (no intermediate variable). Any rectangular shape is fine.

Print both tensors and their `.shape`.


In [ ]:
# 2-D from a list
data = [[1, 2], [3, 4]]
tensor_2d = torch.tensor(data)

# 3-D in one shot — shape (3, 2, 2)
tensor_3d = torch.tensor([[[1, 2], [3, 4]],
                          [[5, 6], [7, 8]],
                          [[9, 10], [11, 12]]])

print(tensor_2d)
print("2d shape", tuple(tensor_2d.shape))
print(tensor_3d)
print("3d shape", tuple(tensor_3d.shape))


## 2. Factory helpers — zeros, ones, arange, linspace, like

**Task.**

1. `zeros` — a 2×3 float tensor.
2. `arange` — integers **1 through 10 inclusive** (so the exclusive end is 11).
3. `linspace` — 10 evenly spaced values from 2 to 4 **inclusive**.
4. Build a reference `x = torch.tensor([[1, 2, 3], [4, 5, 6]])` and make `zeros_like` / `ones_like` copies.

`arange` excludes the end; `linspace` includes both ends. Default dtype for `zeros`/`ones` is `float32`; `arange` of ints is `int64`.


In [ ]:
zeros_23 = torch.zeros(2, 3)
one_to_ten = torch.arange(1, 11)
grid_2_4 = torch.linspace(2, 4, steps=10)

x = torch.tensor([[1, 2, 3], [4, 5, 6]])
zeros_like_x = torch.zeros_like(x)
ones_like_x = torch.ones_like(x)

print(zeros_23)
print(one_to_ten)
print(grid_2_4)
print("like dtype follows x:", zeros_like_x.dtype)
print(zeros_like_x)
print(ones_like_x)


## 3. Load Iris and encode the target

150 rows, 50 of each species. Four measurements in centimetres.

**Task.** Read `data/iris.csv`. Build:

- `X` — NumPy array of the four numeric columns, `float32`
- `y` — integer codes `setosa=0`, `versicolor=1`, `virginica=2`, `int64`
- print `X.shape`, the class counts, and the first three rows


In [ ]:
iris = pd.read_csv("data/iris.csv")
print(iris.head(3))
print(iris["species"].value_counts())

feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
species_to_idx = {"setosa": 0, "versicolor": 1, "virginica": 2}

X = iris[feature_cols].to_numpy(dtype=np.float32)
y = iris["species"].map(species_to_idx).to_numpy(dtype=np.int64)

print("X", X.shape, "y", y.shape, y.dtype)
print("counts", {i: int((y == i).sum()) for i in range(3)})
print(X[:3])


## 4. Stratified 80/20 split, then wrap as tensors

sklearn is **not** required. A class-balanced shuffle is enough (50 per species → 10 test each).

**Task.**

1. Write a small `train_test_split` (or use `PTIntro.train_test_split`).
2. Convert `X_*` to `torch.float32` and `y_*` to `torch.long`.
3. Print shapes. You should see about `(120, 4)` / `(30, 4)`.


In [ ]:
def train_test_split(X, y, test_size=0.2, random_state=42):
    rng = np.random.RandomState(random_state)
    tr, te = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_te = int(round(len(idx) * test_size))
        te.append(idx[:n_te])
        tr.append(idx[n_te:])
    tr = np.concatenate(tr); te = np.concatenate(te)
    rng.shuffle(tr); rng.shuffle(te)
    return X[tr], X[te], y[tr], y[te]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print("train", tuple(X_train_t.shape), tuple(y_train_t.shape))
print("test ", tuple(X_test_t.shape),  tuple(y_test_t.shape))
print("test class counts", y_test_t.bincount().tolist())


## 5. Define `IrisModel`

Subclass `nn.Module`.

- `__init__(self, input_size, hidden_size, output_size)`
- call `super().__init__()` (or `super(IrisModel, self).__init__()`)
- `self.fc1 = nn.Linear(input_size, hidden_size)`
- `self.fc2 = nn.Linear(hidden_size, output_size)`
- `forward`: ReLU after `fc1`, raw logits from `fc2` (no softmax)

Then instantiate with `input_size=4`, `hidden_size=16`, `output_size=3`.


In [ ]:
class IrisModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

input_size = X.shape[1]
output_size = 3
model = IrisModel(input_size, 16, output_size)
print(model)
print("n params", sum(p.numel() for p in model.parameters()))


## 6. Train — loss, optimizer, loop

Use:

- `criterion = nn.CrossEntropyLoss()`
- `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)`
- 100 epochs, full batch (the whole train tensor each step)

Each epoch, in this order:

1. `optimizer.zero_grad()`
2. `y_pred = model(X_train_t)`
3. `loss = criterion(y_pred, y_train_t)`
4. `loss.backward()`
5. `optimizer.step()`

Print the loss every 10 epochs. Store losses in a list so you can plot them.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
epochs = 100
losses = []

for epoch in range(epochs):
    optimizer.zero_grad()
    y_pred = model(X_train_t)
    loss = criterion(y_pred, y_train_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print(f"final loss {losses[-1]:.4f}")
plt.figure(figsize=(6.4, 3.6))
plt.plot(losses, color="#1A5276")
plt.xlabel("epoch"); plt.ylabel("cross-entropy")
plt.title("training loss — Adam lr=0.01, hidden=16")
plt.grid(True, alpha=0.3)
plt.show()


## 7. Evaluate on the held-out 30 flowers

**Task.**

1. `model.eval()`
2. wrap the forward pass in `with torch.no_grad():`
3. `y_test_pred_labels = torch.argmax(y_test_pred, dim=1)`
4. accuracy as a percent

Typical result on this split: **about 93–100%**. Iris is almost linearly separable on petals; a 16-unit hidden layer is more capacity than you need.


In [ ]:
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_t)
    y_test_pred_labels = torch.argmax(y_test_pred, dim=1)

n_correct = (y_test_pred_labels == y_test_t).sum().item()
accuracy = n_correct / len(y_test_t) * 100
print(f"Test accuracy: {accuracy:.2f}%  ({n_correct}/{len(y_test_t)})")

# tiny confusion matrix
idx_to_species = {0: "setosa", 1: "versicolor", 2: "virginica"}
cm = np.zeros((3, 3), dtype=int)
for t, p in zip(y_test_t.numpy(), y_test_pred_labels.numpy()):
    cm[t, p] += 1
print("rows=true, cols=pred\n", cm)


## 8. Alternate code — same result, different spelling

Three swaps that should land within a couple of points of the Module + Adam run:

1. **`nn.Sequential`** instead of a custom class.
2. **SGD** instead of Adam (you may need more epochs or a slightly larger `lr`).
3. **`torch.as_tensor`** on the NumPy arrays (no extra copy when the dtype already matches).

Run the cell. Compare final loss and test accuracy to §6–7.


In [ ]:
Xtr = torch.as_tensor(X_train, dtype=torch.float32)
Xte = torch.as_tensor(X_test,  dtype=torch.float32)
ytr = torch.as_tensor(y_train, dtype=torch.long)
yte = torch.as_tensor(y_test,  dtype=torch.long)

torch.manual_seed(0)
seq = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
opt = torch.optim.SGD(seq.parameters(), lr=0.05)
crit = nn.CrossEntropyLoss()
seq_losses = []
for epoch in range(200):
    opt.zero_grad()
    loss = crit(seq(Xtr), ytr)
    loss.backward()
    opt.step()
    seq_losses.append(loss.item())

seq.eval()
with torch.no_grad():
    pred = seq(Xte).argmax(1)
acc = (pred == yte).float().mean().item() * 100
print(f"Sequential + SGD final loss {seq_losses[-1]:.4f}, test acc {acc:.2f}%")

# NumPy-only baseline: nearest class mean (no torch at all)
means = np.stack([X_train[y_train == k].mean(0) for k in range(3)])
nn_pred = ((X_test[:, None, :] - means[None, :, :]) ** 2).sum(2).argmin(1)
print(f"NumPy class-mean baseline acc { (nn_pred == y_test).mean()*100:.2f}%")


## 9. More practice

Two short drills. Fill them in, then check the solution notebook.

**A. Setosa vs the rest (binary).** Relabel `y_bin = (y == 0).astype(np.int64)`. Use the same split logic, a `4 → 8 → 2` net, and report test accuracy. Setosa is linearly separable — you should land near 100%.

**B. XOR blobs.** `from PTIntro import xor_data`. Fit a `2 → 8 → 2` net for 300 epochs, Adam `lr=0.05`. Then try a *linear* net (`2 → 2`, no hidden ReLU). The linear net should hover near 50%; the hidden net should climb well above that. This is the whole point of the nonlinearity.


In [ ]:
# A. binary setosa vs rest
y_bin = (y == 0).astype(np.int64)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X, y_bin, test_size=0.2, random_state=0)
Xb_tr_t = torch.tensor(Xb_tr); Xb_te_t = torch.tensor(Xb_te)
yb_tr_t = torch.tensor(yb_tr, dtype=torch.long)
yb_te_t = torch.tensor(yb_te, dtype=torch.long)

class BinaryNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.fc2 = nn.Linear(8, 2)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

torch.manual_seed(0)
bin_model = BinaryNet()
opt = torch.optim.Adam(bin_model.parameters(), lr=0.02)
crit = nn.CrossEntropyLoss()
for _ in range(80):
    opt.zero_grad()
    crit(bin_model(Xb_tr_t), yb_tr_t).backward()
    opt.step()
bin_model.eval()
with torch.no_grad():
    acc_b = (bin_model(Xb_te_t).argmax(1) == yb_te_t).float().mean().item()
print(f"A. setosa-vs-rest test acc {acc_b*100:.1f}%")

# B. XOR
from PTIntro import xor_data
Xx, yx = xor_data(n=400, seed=1, noise=0.08)
Xtr, Xte = torch.tensor(Xx[:-80]), torch.tensor(Xx[-80:])
ytr, yte = torch.tensor(yx[:-80], dtype=torch.long), torch.tensor(yx[-80:], dtype=torch.long)

def fit(model, epochs=300, lr=0.05):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad()
        crit(model(Xtr), ytr).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

torch.manual_seed(1)
hidden = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
linear = nn.Linear(2, 2)
print(f"B. XOR hidden-ReLU acc {fit(hidden)*100:.1f}%")
print(f"B. XOR linear acc      {fit(linear)*100:.1f}%  (expect ~50%)")


## 10. Simulation — turn three knobs

Change `HIDDEN`, `LR`, `EPOCHS`, and `LABEL_NOISE`, then re-run.

What you should see on this dataset:

- `LR = 0.001` often underfits in 100 epochs (loss still falling).
- `LR = 0.05` usually hits ≥ 96% with hidden ≥ 8.
- `LABEL_NOISE` up to ~0.2 barely moves test accuracy — petals already separate setosa cleanly, and the other two classes have a small overlap that noise does not create.
- `HIDDEN = 4` is already enough. Iris is not a capacity problem.

This cell uses `PTIntro.train_iris` so you are not re-typing the loop.


In [ ]:
from PTIntro import train_iris

HIDDEN = 16          # try 4, 8, 16, 32
LR = 0.01            # try 0.001, 0.01, 0.05
EPOCHS = 100         # try 30, 100, 300
LABEL_NOISE = 0.0    # try 0.0, 0.1, 0.3  (fraction of train labels flipped)
SEED = 42

result = train_iris(
    X_train, y_train, X_test, y_test,
    hidden_size=HIDDEN, lr=LR, epochs=EPOCHS,
    seed=SEED, label_noise=LABEL_NOISE,
)
print(f"hidden={HIDDEN} lr={LR} epochs={EPOCHS} noise={LABEL_NOISE}")
print(f"final loss {result.losses[-1]:.4f}  test acc {result.test_acc*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))
axes[0].plot(result.losses, color="#1A5276")
axes[0].set_title("loss")
axes[0].set_xlabel("epoch")
axes[0].grid(True, alpha=0.3)

# small grid around the current knobs
grid_h = [4, 8, 16, 32]
grid_lr = [0.001, 0.01, 0.05]
heat = np.zeros((len(grid_h), len(grid_lr)))
for i, h in enumerate(grid_h):
    for j, lr in enumerate(grid_lr):
        heat[i, j] = train_iris(
            X_train, y_train, X_test, y_test,
            hidden_size=h, lr=lr, epochs=EPOCHS, seed=0,
        ).test_acc
im = axes[1].imshow(heat, cmap="YlGn", vmin=0.5, vmax=1.0)
axes[1].set_xticks(range(len(grid_lr)), [str(v) for v in grid_lr])
axes[1].set_yticks(range(len(grid_h)), [str(v) for v in grid_h])
axes[1].set_xlabel("lr"); axes[1].set_ylabel("hidden")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        axes[1].text(j, i, f"{heat[i,j]:.0%}", ha="center", va="center", fontsize=8)
axes[1].set_title(f"acc heatmap @ {EPOCHS} epochs")
fig.colorbar(im, ax=axes[1], fraction=0.046)
fig.tight_layout()
plt.show()


## 11. What this model can and cannot do

**Can**
- Show the tensor API you will use in every later PyTorch lab.
- Separate three Iris species from four measurements at high accuracy.
- Make the training-loop contract muscle memory (`zero_grad` → forward → loss → `backward` → `step`).
- Demonstrate why a hidden ReLU exists (XOR drill).

**Cannot / should not**
- This is **not** a production flower identifier. 150 rows, four hand-picked features, no image input.
- Full-batch Adam on 120 rows is not how you train ImageNet. Use `DataLoader` + GPU when *n* or *p* grows.
- Accuracy on Iris is a weak signal — a class-mean baseline is already strong. Judge the *loop*, not the leaderboard.
- Do not attach this net to a field-guide app and eat a mushroom. Wrong domain.

**Four audiences (see `PTIntro_Project_Memo.docx`)**
- Researcher: dynamic graph + autograd tape.
- Engineer: `Module` / Sequential / eval-mode contract.
- Executive: small prototype, minutes on CPU, not a product claim.
- Nonspecialist: the network votes for the closest of three named flowers.
